# The SMO Algorithm: Sequential Minimal Optimization

**Difficulty:** ⭐⭐⭐⭐⭐ | **Estimated Time:** 2.5 hours | **Week 10 — Notebook 10**

---

## Why Does This Matter?

You now know that SVM solves a quadratic program (QP) with n variables (one $\alpha_i$ per training point) and n+1 constraints. For n=10,000 training points, a naive QP solver needs to invert a 10,000×10,000 matrix — **O(n³) time and O(n²) memory**. That's about 10¹² operations and 800 GB of RAM for the kernel matrix alone.

**SMO (Sequential Minimal Optimization)**, introduced by John Platt at Microsoft Research in 1998, solved this problem by decomposing the QP into the **smallest possible subproblems**: pairs of two variables at a time. Each 2-variable subproblem has a closed-form analytical solution — no iterative solver needed. The result was a practical SVM algorithm that could train on datasets with hundreds of thousands of points.

**Why it matters today:** Even with modern hardware and solvers (LIBSVM, sklearn), understanding SMO helps you:
- Debug convergence issues
- Understand what KKT conditions mean in practice
- Reason about why SVMs scale the way they do
- Appreciate modern variants (SGD-based SVMs, kernel approximations)

## Real-World Analogy: Balancing a Department Budget

Imagine you manage a university with K departments. The total budget is fixed at \$10M. You need to re-allocate funds across departments to maximize some performance metric (research output, student satisfaction, etc.).

**The constraint:** The total across all departments must always equal \$10M. You can't just add money to one department — you must take it from another.

**If you adjust departments one at a time:** Impossible. Adding \$100K to Computer Science immediately violates the total budget constraint, because now you have \$10.1M allocated.

**The solution:** Adjust **two departments at a time**. Add \$100K to CS, remove \$100K from Administration. The total stays at \$10M. This is the minimum number of departments you can touch while keeping the budget balanced.

**SMO does exactly this for the SVM dual:**
- The budget = the SVM constraint: $\sum_i \alpha_i y_i = 0$
- The department allocations = the Lagrange multipliers $\alpha_1, \alpha_2, \ldots, \alpha_n$
- **Adjusting just one $\alpha_i$** would violate $\sum_i \alpha_i y_i = 0$
- **Adjusting two $\alpha_i$ and $\alpha_j$ jointly** can maintain the constraint
- SMO picks the best pair to adjust at each step and solves for the optimal values analytically

## Plain English: How SMO Works

**The SVM dual problem:**
$$\max_{\alpha} \sum_i \alpha_i - \frac{1}{2}\sum_{i,j} \alpha_i \alpha_j y_i y_j K(x_i, x_j)$$
$$\text{subject to: } 0 \leq \alpha_i \leq C, \quad \sum_i \alpha_i y_i = 0$$

**SMO's approach — three steps, repeated:**

**Step 1 — Find a violated $\alpha$:** Look for an $\alpha_i$ that violates the KKT conditions (it's not at its optimal value yet). These are the "badly allocated" departments.

**Step 2 — Pick a partner $\alpha_j$:** Choose the second $\alpha_j$ that, when updated together with $\alpha_i$, will make the largest improvement (maximize |E_i - E_j|, i.e., pick the partner with the most opposite "error").

**Step 3 — Solve analytically:** Given the constraint $\alpha_i y_i + \alpha_j y_j = \text{constant}$, there is only **one degree of freedom** — the 2-variable QP reduces to a 1-variable quadratic, which has a closed-form solution. Clip to the box $[0, C] \times [0, C]$. Update the bias term b. Repeat.

**Convergence:** Keep going until no $\alpha_i$ violates KKT by more than tolerance $\tau$. At that point, every $\alpha_i$ is at its optimal value and the SVM is fully trained.

---

## KKT Conditions for SVM

A point $x_i$ satisfies the KKT conditions when:

| Case | Condition | Meaning |
|------|-----------|----------|
| $\alpha_i = 0$ | $y_i f(x_i) \geq 1$ | Inside or on correct side of margin |
| $0 < \alpha_i < C$ | $y_i f(x_i) = 1$ | Exactly on the margin (support vector) |
| $\alpha_i = C$ | $y_i f(x_i) \leq 1$ | Inside the margin or misclassified |

KKT violation: $\alpha_i$ is not in the correct range for its $y_i f(x_i)$ value.

---

## The 2-Variable Update Formula

Given the current errors $E_i = f(x_i) - y_i$ and the kernel values $K_{ij} = K(x_i, x_j)$:

$$\eta = K_{ii} + K_{jj} - 2K_{ij} \quad (\text{must be} > 0)$$

$$\alpha_j^{\text{new}} = \alpha_j + \frac{y_j(E_i - E_j)}{\eta}$$

**Clip to bounds** $[L, H]$:
- If $y_i = y_j$: $L = \max(0, \alpha_i + \alpha_j - C)$, $H = \min(C, \alpha_i + \alpha_j)$
- If $y_i \neq y_j$: $L = \max(0, \alpha_j - \alpha_i)$, $H = \min(C, C + \alpha_j - \alpha_i)$

$$\alpha_i^{\text{new}} = \alpha_i + y_i y_j (\alpha_j^{\text{old}} - \alpha_j^{\text{new}})$$

**Update bias:**
$$b_1 = b - E_i - y_i(\alpha_i^{\text{new}} - \alpha_i^{\text{old}})K_{ii} - y_j(\alpha_j^{\text{new}} - \alpha_j^{\text{old}})K_{ij}$$
$$b_2 = b - E_j - y_i(\alpha_i^{\text{new}} - \alpha_i^{\text{old}})K_{ij} - y_j(\alpha_j^{\text{new}} - \alpha_j^{\text{old}})K_{jj}$$
$$b = \frac{b_1 + b_2}{2}$$

In [ ]:
# =============================================================================
# SECTION 1: Imports & Setup
# =============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.svm import SVC
from sklearn.datasets import make_classification, make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

print("Libraries loaded. Ready to implement SMO from scratch.")
print("Theme: Spam email classification (document analysis)")

In [ ]:
# =============================================================================
# SECTION 2: Spam Email Dataset (Linearly Separable, 2D)
# =============================================================================
# Scenario: Classify emails as spam (+1) or legitimate (-1) using two features:
#   - feature_1: frequency of spam-indicator words (normalized)
#   - feature_2: sender reputation score (normalized)

np.random.seed(42)
n_per_class = 25

# Legitimate emails: low word frequency, higher reputation
X_legit = np.column_stack([
    np.random.normal(-1.5, 0.6, n_per_class),  # low spam word freq
    np.random.normal( 1.5, 0.6, n_per_class),  # high sender reputation
])
y_legit = np.full(n_per_class, -1)

# Spam emails: high word frequency, lower reputation
X_spam = np.column_stack([
    np.random.normal( 1.5, 0.6, n_per_class),  # high spam word freq
    np.random.normal(-1.5, 0.6, n_per_class),  # low sender reputation
])
y_spam = np.full(n_per_class, +1)

X = np.vstack([X_legit, X_spam])
y = np.concatenate([y_legit, y_spam])

# Shuffle
shuffle_idx = np.random.permutation(len(X))
X, y = X[shuffle_idx], y[shuffle_idx]

print("Email Spam Classification Dataset")
print("="*40)
print(f"Total samples:      {len(X)}")
print(f"Spam (+1):          {(y == 1).sum()}")
print(f"Legitimate (-1):    {(y == -1).sum()}")
print(f"Features:           [spam_word_freq, sender_reputation]")
print()

# Visualize
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[y==-1, 0], X[y==-1, 1], color='#3498db', s=80,
           edgecolors='black', linewidths=0.7, label='Legitimate (-1)', zorder=3)
ax.scatter(X[y==+1, 0], X[y==+1, 1], color='#e74c3c', s=80, marker='^',
           edgecolors='black', linewidths=0.7, label='Spam (+1)',       zorder=3)
ax.set_xlabel('Spam Word Frequency', fontsize=12)
ax.set_ylabel('Sender Reputation Score', fontsize=12)
ax.set_title('Email Spam Classification Dataset\n(2D, linearly separable)', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SECTION 3: Simplified SMO Implementation
# =============================================================================
# Full implementation with convergence tracking.
# We track: alpha values, number of changed alphas, KKT violations per pass.

class SimpleSMO:
    """
    Simplified SMO for binary SVM classification.
    Labels must be in {-1, +1}.
    Uses random selection for the second alpha (simplified heuristic).
    Tracks convergence metrics at each iteration.
    """

    def __init__(self, C=1.0, tol=1e-3, max_iter=100, random_state=42):
        self.C           = C
        self.tol         = tol
        self.max_iter    = max_iter
        self.random_state = random_state

    def _kernel(self, x1, x2):
        """Linear kernel: K(x1, x2) = x1 · x2"""
        return x1 @ x2

    def _decision_function(self, x):
        """f(x) = sum_i alpha_i * y_i * K(x_i, x) + b"""
        kernel_vals = self.X_ @ x  # shape (n,)
        return np.dot(self.alpha_ * self.y_, kernel_vals) + self.b_

    def _compute_kkt_violations(self):
        """Count how many alpha_i violate KKT conditions."""
        violations = 0
        for i in range(len(self.X_)):
            Ei = self._decision_function(self.X_[i]) - self.y_[i]
            yi_Ei = self.y_[i] * Ei
            alpha_i = self.alpha_[i]
            # KKT violation:
            if (yi_Ei < -self.tol and alpha_i < self.C) or \
               (yi_Ei >  self.tol and alpha_i > 0):
                violations += 1
        return violations

    def _dual_objective(self):
        """Compute dual objective (we want to maximize this)."""
        # W(alpha) = sum_i alpha_i - 0.5 * sum_{i,j} alpha_i alpha_j y_i y_j K(x_i,x_j)
        K = self.X_ @ self.X_.T  # linear kernel matrix
        ayay = np.outer(self.alpha_ * self.y_, self.alpha_ * self.y_)
        return self.alpha_.sum() - 0.5 * (ayay * K).sum()

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n = len(X)
        self.X_     = X.copy()
        self.y_     = y.astype(float).copy()
        self.alpha_ = np.zeros(n)
        self.b_     = 0.0

        # Tracking history
        self.history_changed_    = []   # num changed alphas per outer iteration
        self.history_violations_ = []   # KKT violations per outer iteration
        self.history_objective_  = []   # dual objective per outer iteration
        self.pairs_selected_     = []   # (i, j) pairs selected for update
        self.alpha_history_      = []   # snapshot of all alphas per outer iteration

        for iteration in range(self.max_iter):
            num_changed = 0

            for i in range(n):
                Ei = self._decision_function(self.X_[i]) - self.y_[i]

                # Check KKT violation for alpha_i
                if (self.y_[i] * Ei < -self.tol and self.alpha_[i] < self.C) or \
                   (self.y_[i] * Ei >  self.tol and self.alpha_[i] > 0):

                    # --- Heuristic: pick j that maximizes |Ei - Ej| ---
                    errors = np.array([self._decision_function(self.X_[k]) - self.y_[k]
                                       for k in range(n)])
                    # Maximize |Ei - Ej|
                    diffs = np.abs(Ei - errors)
                    diffs[i] = -1  # exclude i itself
                    j = int(np.argmax(diffs))
                    Ej = errors[j]

                    self.pairs_selected_.append((i, j))

                    alpha_i_old = self.alpha_[i]
                    alpha_j_old = self.alpha_[j]

                    # Compute bounds L, H
                    if self.y_[i] == self.y_[j]:
                        L = max(0, self.alpha_[j] + self.alpha_[i] - self.C)
                        H = min(self.C, self.alpha_[j] + self.alpha_[i])
                    else:
                        L = max(0, self.alpha_[j] - self.alpha_[i])
                        H = min(self.C, self.C + self.alpha_[j] - self.alpha_[i])

                    if L >= H:
                        continue

                    # Compute eta (second-order step size denominator)
                    Kii = self._kernel(self.X_[i], self.X_[i])
                    Kjj = self._kernel(self.X_[j], self.X_[j])
                    Kij = self._kernel(self.X_[i], self.X_[j])
                    eta = Kii + Kjj - 2 * Kij

                    if eta <= 0:
                        continue

                    # Update alpha_j
                    self.alpha_[j] += self.y_[j] * (Ei - Ej) / eta
                    self.alpha_[j]  = np.clip(self.alpha_[j], L, H)

                    if abs(self.alpha_[j] - alpha_j_old) < 1e-5:
                        continue

                    # Update alpha_i (to maintain sum(alpha * y) = 0)
                    self.alpha_[i] += self.y_[i] * self.y_[j] * (alpha_j_old - self.alpha_[j])

                    # Update bias b
                    b1 = (self.b_ - Ei
                          - self.y_[i] * (self.alpha_[i] - alpha_i_old) * Kii
                          - self.y_[j] * (self.alpha_[j] - alpha_j_old) * Kij)
                    b2 = (self.b_ - Ej
                          - self.y_[i] * (self.alpha_[i] - alpha_i_old) * Kij
                          - self.y_[j] * (self.alpha_[j] - alpha_j_old) * Kjj)

                    if 0 < self.alpha_[i] < self.C:
                        self.b_ = b1
                    elif 0 < self.alpha_[j] < self.C:
                        self.b_ = b2
                    else:
                        self.b_ = (b1 + b2) / 2

                    num_changed += 1

            # Record metrics for this outer iteration
            self.history_changed_.append(num_changed)
            self.history_violations_.append(self._compute_kkt_violations())
            self.history_objective_.append(self._dual_objective())
            self.alpha_history_.append(self.alpha_.copy())

            if num_changed == 0:
                print(f"Converged at iteration {iteration + 1}.")
                break

        self.support_mask_   = self.alpha_ > 1e-5
        self.n_support_      = self.support_mask_.sum()
        self.alpha_history_  = np.array(self.alpha_history_)
        return self

    def predict(self, X):
        return np.sign([self._decision_function(x) for x in X])

print("SimpleSMO class defined.")
print("Features:")
print("  - Linear kernel")
print("  - Max |Ei - Ej| working set selection")
print("  - Full KKT violation monitoring")
print("  - Dual objective tracking")
print("  - Alpha history for convergence plots")

In [ ]:
# =============================================================================
# SECTION 4: Train SimpleSMO on Email Dataset
# =============================================================================

smo = SimpleSMO(C=1.0, tol=1e-3, max_iter=50, random_state=42)
smo.fit(X, y)

y_pred_smo = smo.predict(X)
acc_smo = accuracy_score(y, y_pred_smo)

print(f"SimpleSMO Training Complete")
print(f"="*40)
print(f"Training accuracy:  {acc_smo:.4f}")
print(f"Support vectors:    {smo.n_support_}")
print(f"Bias (b):           {smo.b_:.4f}")
print(f"Total iterations:   {len(smo.history_changed_)}")
print()

# Show the learned alpha values for SVs
sv_indices = np.where(smo.support_mask_)[0]
print(f"Support Vector Indices and α Values:")
for idx in sv_indices:
    print(f"  Point {idx:3d}: α={smo.alpha_[idx]:.4f}  y={int(y[idx]):+d}  x={X[idx]}")

In [ ]:
# =============================================================================
# SECTION 5: Convergence Tracking — Changed Alphas & Objective Value
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

iters = np.arange(1, len(smo.history_changed_) + 1)

# --- Left: Changed alphas per iteration ---
ax = axes[0]
ax.bar(iters, smo.history_changed_, color='#3498db', alpha=0.8, edgecolor='white')
ax.set_xlabel('Outer Iteration', fontsize=12)
ax.set_ylabel('Number of Changed α Values', fontsize=12)
ax.set_title('SMO Convergence:\nChanged Alphas per Iteration', fontsize=12, fontweight='bold')
ax.axhline(0, color='red', linestyle='--', linewidth=1.5, label='Convergence (0 changes)')
ax.legend(fontsize=10)

# --- Middle: KKT violations per iteration ---
ax2 = axes[1]
ax2.plot(iters, smo.history_violations_, 'o-', color='#e74c3c',
         linewidth=2.5, markersize=7, label='KKT violations')
ax2.fill_between(iters, smo.history_violations_, alpha=0.15, color='#e74c3c')
ax2.set_xlabel('Outer Iteration', fontsize=12)
ax2.set_ylabel('KKT Violations Remaining', fontsize=12)
ax2.set_title('SMO Convergence:\nKKT Violations Decreasing', fontsize=12, fontweight='bold')
ax2.axhline(0, color='green', linestyle='--', linewidth=1.5, label='KKT satisfied')
ax2.legend(fontsize=10)

# --- Right: Dual objective per iteration ---
ax3 = axes[2]
ax3.plot(iters, smo.history_objective_, 's-', color='#2ecc71',
         linewidth=2.5, markersize=7, label='Dual objective W(α)')
ax3.fill_between(iters, smo.history_objective_,
                  min(smo.history_objective_) - 0.1,
                  alpha=0.1, color='#2ecc71')
ax3.set_xlabel('Outer Iteration', fontsize=12)
ax3.set_ylabel('Dual Objective W(α)', fontsize=12)
ax3.set_title('SMO Convergence:\nDual Objective Increasing to Optimum', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)

plt.suptitle('SMO Algorithm: Convergence Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"SMO converged in {len(smo.history_changed_)} outer iterations.")
print(f"Initial KKT violations: {smo.history_violations_[0]}")
print(f"Final   KKT violations: {smo.history_violations_[-1]}")
print(f"Dual objective: {smo.history_objective_[0]:.4f} → {smo.history_objective_[-1]:.4f}")

In [ ]:
# =============================================================================
# SECTION 6: Compare SimpleSMO vs sklearn SVC
# =============================================================================

sklearn_svc = SVC(kernel='linear', C=1.0)
sklearn_svc.fit(X, y)
y_pred_sklearn = sklearn_svc.predict(X)
acc_sklearn = accuracy_score(y, y_pred_sklearn)

print("Comparison: SimpleSMO vs sklearn SVC")
print("="*50)
print(f"{'Metric':<30} {'SimpleSMO':>12} {'sklearn SVC':>12}")
print("-"*56)
print(f"{'Training Accuracy':<30} {acc_smo:>12.4f} {acc_sklearn:>12.4f}")
print(f"{'# Support Vectors':<30} {smo.n_support_:>12} {sklearn_svc.n_support_.sum():>12}")
print(f"{'Bias b':<30} {smo.b_:>12.4f} {-sklearn_svc.intercept_[0]:>12.4f}")

# Check if predictions agree
agreement = (y_pred_smo == y_pred_sklearn).mean()
print(f"{'Prediction agreement':<30} {agreement:>12.4f}")
print()

# Visualize decision boundaries side by side
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                      np.linspace(y_min, y_max, 200))
grid_pts = np.column_stack([xx.ravel(), yy.ravel()])

# SMO decision surface
z_smo = np.array([smo._decision_function(pt) for pt in grid_pts]).reshape(xx.shape)
# Sklearn decision surface
z_sk  = sklearn_svc.decision_function(grid_pts).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, z, title, model_type in zip(
        axes,
        [z_smo, z_sk],
        ['SimpleSMO (from scratch)', 'sklearn SVC (LIBSVM)'],
        ['smo', 'sklearn']):

    ax.contourf(xx, yy, z, levels=[-np.inf, 0, np.inf],
                colors=['#aed6f1', '#f1948a'], alpha=0.3)
    ax.contour(xx, yy, z, levels=[-1, 0, 1],
               colors=['#2980b9', '#2c3e50', '#c0392b'],
               linewidths=[1.5, 2.5, 1.5], linestyles=['--', '-', '--'])

    # Points
    ax.scatter(X[y==-1, 0], X[y==-1, 1], c='#3498db', s=70,
               edgecolors='black', linewidths=0.7, label='Legitimate', zorder=3)
    ax.scatter(X[y==+1, 0], X[y==+1, 1], c='#e74c3c', s=70, marker='^',
               edgecolors='black', linewidths=0.7, label='Spam',       zorder=3)

    # Highlight SVs
    if model_type == 'smo':
        sv_idx = np.where(smo.support_mask_)[0]
        ax.scatter(X[sv_idx, 0], X[sv_idx, 1], s=200, facecolors='none',
                   edgecolors='#f39c12', linewidths=2.5, label=f'SVs ({len(sv_idx)})', zorder=4)
    else:
        sv_idx = sklearn_svc.support_
        ax.scatter(X[sv_idx, 0], X[sv_idx, 1], s=200, facecolors='none',
                   edgecolors='#f39c12', linewidths=2.5, label=f'SVs ({len(sv_idx)})', zorder=4)

    ax.set_xlabel('Spam Word Frequency', fontsize=11)
    ax.set_ylabel('Sender Reputation', fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)

plt.suptitle('Decision Boundaries: SimpleSMO vs sklearn SVC', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SECTION 7: Alpha Values Converging Over Iterations
# =============================================================================
# Show how the alpha values of the final support vectors evolve over SMO iterations.

sv_final_indices = np.where(smo.alpha_ > 1e-5)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: alpha trajectories for ALL support vectors
ax = axes[0]
iters = np.arange(1, len(smo.alpha_history_) + 1)
colors_sv = plt.cm.Set2(np.linspace(0, 1, len(sv_final_indices)))

for sv_i, col in zip(sv_final_indices, colors_sv):
    alpha_traj = smo.alpha_history_[:, sv_i]
    ax.plot(iters, alpha_traj, 'o-', color=col, linewidth=2,
            markersize=6, label=f'α[{sv_i}] (y={int(y[sv_i]):+d})')

ax.axhline(smo.C, color='gray', linestyle='--', linewidth=1.2, alpha=0.7, label=f'C={smo.C}')
ax.axhline(0,     color='gray', linestyle=':',  linewidth=1.2, alpha=0.7)
ax.set_xlabel('Outer Iteration', fontsize=12)
ax.set_ylabel('α Value', fontsize=12)
ax.set_title('Alpha Values of Support Vectors\nConverging Over SMO Iterations',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc='upper left')

# Right: heatmap of all alpha values over time (full picture)
ax2 = axes[1]
# Select interesting indices: final SVs + a few non-SVs
non_sv_sample = np.where(smo.alpha_ <= 1e-5)[0][:5]
show_indices  = np.concatenate([sv_final_indices, non_sv_sample])
alpha_subset  = smo.alpha_history_[:, show_indices].T

im = ax2.imshow(alpha_subset, aspect='auto', cmap='YlOrRd',
                vmin=0, vmax=smo.C)
plt.colorbar(im, ax=ax2, label='α value')
ax2.set_xlabel('SMO Outer Iteration', fontsize=11)
ax2.set_ylabel('Training Point Index', fontsize=11)
ax2.set_yticks(range(len(show_indices)))
labels = [f'SV:{idx}' for idx in sv_final_indices] + [f'Non-SV:{idx}' for idx in non_sv_sample]
ax2.set_yticklabels(labels, fontsize=9)
ax2.set_title('Heatmap of α Values Over Iterations\n(warm=large α, cool=zero)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Observation:")
print(f"  Final support vectors: {list(sv_final_indices)}")
print(f"  Their final alpha values: {smo.alpha_[sv_final_indices].round(4)}")
print(f"  Verification: sum(alpha_i * y_i) = {np.dot(smo.alpha_, y):.6f} (should be ~0)")

In [ ]:
# =============================================================================
# SECTION 8: KKT Violations — Detailed View
# =============================================================================
# Show KKT status for every training point at start, mid, and end of training.

def check_kkt_status(X, y, alpha, b, tol=1e-3, C=1.0):
    """
    Return KKT status for each point:
      'satisfied'  - KKT conditions met
      'violated'   - KKT conditions not met
    """
    K = X @ X.T
    f = K @ (alpha * y) + b
    statuses = []
    for i in range(len(X)):
        yf = y[i] * f[i]
        a  = alpha[i]
        if (a < tol and yf >= 1 - tol):          # alpha~0, inside margin
            statuses.append('satisfied')
        elif (a > C - tol and yf <= 1 + tol):    # alpha~C, on/outside margin
            statuses.append('satisfied')
        elif (tol <= a <= C - tol and abs(yf - 1) <= tol):  # 0 < alpha < C, on margin
            statuses.append('satisfied')
        else:
            statuses.append('violated')
    return statuses

# Check at three moments: iteration 0, 1, final
checkpoints = [0, min(1, len(smo.alpha_history_)-1), len(smo.alpha_history_)-1]
checkpoint_labels = ['Start (iter 1)', 'After iter 2', f'Final (iter {len(smo.alpha_history_)})']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, cp, cp_label in zip(axes, checkpoints, checkpoint_labels):
    alpha_cp = smo.alpha_history_[cp]
    # Recompute b at that checkpoint (approximate: use current final b for simplicity)
    # For a full implementation we'd track b too — here we use final b
    statuses = check_kkt_status(X, y, alpha_cp, smo.b_,
                                  tol=smo.tol, C=smo.C)

    colors_pts = ['#e74c3c' if s == 'violated' else '#2ecc71' for s in statuses]
    n_violated  = statuses.count('violated')

    ax.scatter(X[:, 0], X[:, 1], c=colors_pts, s=80,
               edgecolors='black', linewidths=0.7, zorder=3)

    # Add legend patches manually
    ax.scatter([], [], c='#2ecc71', s=80, edgecolors='black', linewidths=0.7,
               label=f'KKT satisfied ({len(statuses)-n_violated})')
    ax.scatter([], [], c='#e74c3c', s=80, edgecolors='black', linewidths=0.7,
               label=f'KKT violated ({n_violated})')

    ax.set_title(f'{cp_label}\nViolations: {n_violated}/{len(X)}',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Spam Word Frequency', fontsize=10)
    ax.set_ylabel('Sender Reputation', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('KKT Violation Status Over SMO Training\n(Red = violated, Green = satisfied)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("KKT violations count per iteration:")
for it, viol in enumerate(smo.history_violations_):
    bar = '#' * int(viol / max(smo.history_violations_) * 30) if smo.history_violations_ else ''
    print(f"  Iter {it+1:2d}: {viol:3d} violations  {bar}")

In [ ]:
# =============================================================================
# SECTION 9: Working Set Selection — Which Pairs Get Selected?
# =============================================================================
# Visualize which pairs (i, j) are selected most often during SMO.
# Hypothesis: support vectors (on the margin) should be selected most.

pairs = smo.pairs_selected_

# Count how often each point appears as i or j
point_selection_count = np.zeros(len(X))
for (i, j) in pairs:
    point_selection_count[i] += 1
    point_selection_count[j] += 1

# Pair frequency matrix
pair_matrix = np.zeros((len(X), len(X)))
for (i, j) in pairs:
    pair_matrix[i, j] += 1
    pair_matrix[j, i] += 1  # symmetric

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: Bubble plot — bubble size = selection frequency
ax = axes[0]
bubble_sizes = (point_selection_count / point_selection_count.max() * 300) + 20
ax.scatter(X[y==-1, 0], X[y==-1, 1],
           s=bubble_sizes[y==-1], c='#3498db',
           edgecolors='black', linewidths=0.7, alpha=0.7,
           label='Legitimate', zorder=3)
ax.scatter(X[y==+1, 0], X[y==+1, 1],
           s=bubble_sizes[y==+1], c='#e74c3c', marker='^',
           edgecolors='black', linewidths=0.7, alpha=0.7,
           label='Spam', zorder=3)

# Mark final SVs with a ring
sv_mask = smo.support_mask_
ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=350, facecolors='none',
           edgecolors='#f39c12', linewidths=2.5, label='Final SVs', zorder=4)

ax.set_xlabel('Spam Word Frequency', fontsize=11)
ax.set_ylabel('Sender Reputation', fontsize=11)
ax.set_title('Working Set Selection Frequency\n(bubble size = #times selected as i or j)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Right: Bar chart — top 15 most selected points
ax2 = axes[1]
top_n  = 15
top_idx = np.argsort(point_selection_count)[::-1][:top_n]
top_cnt = point_selection_count[top_idx]
top_is_sv = smo.support_mask_[top_idx]

bar_colors = ['#f39c12' if sv else '#95a5a6' for sv in top_is_sv]
ax2.barh([f'Point {idx}' + (' (SV)' if sv else '')
          for idx, sv in zip(top_idx, top_is_sv)],
         top_cnt, color=bar_colors, edgecolor='black', linewidth=0.7)
ax2.set_xlabel('Times Selected in Working Set', fontsize=11)
ax2.set_title(f'Top {top_n} Most Selected Points\n(orange = final SVs)',
              fontsize=12, fontweight='bold')
ax2.invert_yaxis()

plt.suptitle('SMO Working Set Selection Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

sv_selection_total    = point_selection_count[smo.support_mask_].sum()
nonsv_selection_total = point_selection_count[~smo.support_mask_].sum()
total_selections      = point_selection_count.sum()
print(f"Total working set selections: {len(pairs)}")
print(f"Selections involving final SVs:     {sv_selection_total:.0f}  ({100*sv_selection_total/total_selections:.1f}%)")
print(f"Selections involving non-SVs:       {nonsv_selection_total:.0f}  ({100*nonsv_selection_total/total_selections:.1f}%)")
print()
print("Key insight: SMO naturally focuses on the points near the decision boundary (SVs).")
print("Interior points (far from the boundary) quickly converge to α=0 and are rarely re-selected.")

In [ ]:
# =============================================================================
# SECTION 10: Speed Comparison — SMO vs scipy QP vs sklearn SVC
# =============================================================================
from scipy.optimize import minimize

def scipy_qp_svm(X, y, C=1.0):
    """Solve SVM dual using scipy minimize (SLSQP)."""
    n = len(X)
    K = X @ X.T  # linear kernel matrix
    yy = np.outer(y, y)

    def neg_dual(alpha):
        return 0.5 * np.dot(alpha, (yy * K) @ alpha) - alpha.sum()

    def neg_dual_grad(alpha):
        return (yy * K) @ alpha - np.ones(n)

    constraints = [{'type': 'eq', 'fun': lambda a: np.dot(a, y)}]
    bounds      = [(0, C)] * n
    alpha0      = np.zeros(n)

    result = minimize(neg_dual, alpha0, jac=neg_dual_grad,
                      method='SLSQP', bounds=bounds, constraints=constraints,
                      options={'maxiter': 1000, 'ftol': 1e-6})
    return result.x

def generate_dataset(n, seed=42):
    np.random.seed(seed)
    X_g = np.vstack([
        np.random.normal([-1.5, 1.5], 0.6, (n//2, 2)),
        np.random.normal([ 1.5,-1.5], 0.6, (n//2, 2))
    ])
    y_g = np.array([-1]*(n//2) + [1]*(n//2), dtype=float)
    return X_g, y_g

sizes = [50, 100, 200, 500]
results_time = {'SimpleSMO': [], 'scipy QP': [], 'sklearn SVC': []}

print(f"{'n':>6} | {'SimpleSMO (s)':>14} | {'scipy QP (s)':>13} | {'sklearn SVC (s)':>15}")
print("-"*58)

for n_pts in sizes:
    X_t, y_t = generate_dataset(n_pts)

    # SimpleSMO
    t0 = time.perf_counter()
    smo_t = SimpleSMO(C=1.0, max_iter=30, random_state=42)
    smo_t.fit(X_t, y_t)
    t_smo = time.perf_counter() - t0
    results_time['SimpleSMO'].append(t_smo)

    # scipy QP (skip for large n due to memory)
    if n_pts <= 200:
        t0 = time.perf_counter()
        scipy_qp_svm(X_t, y_t, C=1.0)
        t_scipy = time.perf_counter() - t0
    else:
        t_scipy = float('nan')
    results_time['scipy QP'].append(t_scipy)

    # sklearn SVC
    t0 = time.perf_counter()
    SVC(kernel='linear', C=1.0).fit(X_t, y_t)
    t_sk = time.perf_counter() - t0
    results_time['sklearn SVC'].append(t_sk)

    print(f"{n_pts:>6} | {t_smo:>14.4f} | {t_scipy:>13.4f} | {t_sk:>15.4f}")

print()
print("Note: scipy QP skipped for n=500 (O(n^3) memory and time issues)")
print("sklearn SVC uses LIBSVM — a highly optimized SMO variant in C++.")

In [ ]:
# =============================================================================
# SECTION 11: Speed Comparison — Visualization
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_sp = ['#3498db', '#e74c3c', '#2ecc71']
markers   = ['o', 's', '^']

# Left: time vs n
ax = axes[0]
for (method, times), col, mk in zip(results_time.items(), colors_sp, markers):
    valid = [(s, t) for s, t in zip(sizes, times) if not (isinstance(t, float) and np.isnan(t))]
    if valid:
        xs, ys = zip(*valid)
        ax.plot(xs, ys, f'{mk}-', color=col, linewidth=2.5, markersize=9, label=method)

ax.set_xlabel('Training Set Size (n)', fontsize=12)
ax.set_ylabel('Training Time (seconds)', fontsize=12)
ax.set_title('Training Time vs Dataset Size', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)

# Right: log-log scale (to see scaling exponent)
ax2 = axes[1]
for (method, times), col, mk in zip(results_time.items(), colors_sp, markers):
    valid = [(s, t) for s, t in zip(sizes, times) if not (isinstance(t, float) and np.isnan(t))]
    if valid:
        xs, ys = zip(*valid)
        ax2.loglog(xs, ys, f'{mk}-', color=col, linewidth=2.5, markersize=9, label=method)

# Reference lines
n_ref = np.array([50, 500], dtype=float)
ax2.loglog(n_ref, 1e-4 * n_ref**2, 'k--', alpha=0.4, linewidth=1.5, label='O(n²) reference')
ax2.loglog(n_ref, 1e-7 * n_ref**3, 'gray', linestyle=':', linewidth=1.5, label='O(n³) reference')

ax2.set_xlabel('Training Set Size (n) — log scale', fontsize=12)
ax2.set_ylabel('Training Time (s) — log scale', fontsize=12)
ax2.set_title('Log-Log Scaling Plot\n(slope = exponent in O(n^k))', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)

plt.suptitle('SMO vs scipy QP vs sklearn SVC — Speed Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Interpretation:")
print("  - SimpleSMO: O(n) to O(n²) per pass — reasonable Python implementation")
print("  - scipy QP:  O(n³) — impractical for large n (also memory O(n²))")
print("  - sklearn SVC: LIBSVM (C++, highly optimized) — fastest in practice")
print("  - Log-log slope ≈ exponent: slope=2 means O(n²), slope=3 means O(n³)")

In [ ]:
# =============================================================================
# SECTION 12: Effect of C on SMO Convergence Speed
# =============================================================================
# Large C → more SVs, more KKT constraints to satisfy → slower convergence

C_values_conv = [0.1, 1.0, 10.0]
conv_results  = {}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, C_val in zip(axes, C_values_conv):
    smo_c = SimpleSMO(C=C_val, tol=1e-3, max_iter=80, random_state=42)
    smo_c.fit(X, y)
    conv_results[C_val] = smo_c

    iters_c = np.arange(1, len(smo_c.history_violations_) + 1)
    ax.plot(iters_c, smo_c.history_violations_, 'o-',
            linewidth=2.5, markersize=7, color='#e74c3c')
    ax.fill_between(iters_c, smo_c.history_violations_, alpha=0.15, color='#e74c3c')
    ax.set_xlabel('Outer Iteration', fontsize=11)
    ax.set_ylabel('KKT Violations', fontsize=11)
    ax.set_title(
        f'C = {C_val}\n'
        f'Converged in {len(smo_c.history_violations_)} iters\n'
        f'SVs = {smo_c.n_support_}',
        fontsize=12, fontweight='bold'
    )
    ax.set_ylim(-1, max(smo_c.history_violations_) * 1.15 + 1)

plt.suptitle('SMO Convergence Speed vs C\n(More violations with larger C → slower convergence)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"{'C':>6} | {'Iterations':>11} | {'Final SVs':>10} | {'Training Acc':>13}")
print("-"*50)
for C_val in C_values_conv:
    smo_c = conv_results[C_val]
    iters_n = len(smo_c.history_violations_)
    n_sv    = smo_c.n_support_
    acc     = accuracy_score(y, smo_c.predict(X))
    print(f"{C_val:>6} | {iters_n:>11} | {n_sv:>10} | {acc:>13.4f}")

In [ ]:
# =============================================================================
# SECTION 13: Visualizing the 2-Variable Update — Step by Step
# =============================================================================
# Show conceptually what happens in one SMO step:
# Hold all alphas fixed, update only (alpha_i, alpha_j).
# The constraint alpha_i*y_i + alpha_j*y_j = const defines a line in alpha space.
# The quadratic objective along that line has a unique maximum.

np.random.seed(42)

# Take first SV pair from our trained SMO
if len(sv_final_indices) >= 2:
    i_demo, j_demo = sv_final_indices[0], sv_final_indices[1]
else:
    i_demo, j_demo = 0, 1

# Fix all alphas at zeros, then show the constrained optimization for (i, j)
alpha_demo    = np.zeros(len(X))
yi, yj        = y[i_demo], y[j_demo]
xi, xj        = X[i_demo], X[j_demo]

# Kernel values
Kii = xi @ xi
Kjj = xj @ xj
Kij = xi @ xj

# Constraint: alpha_i * yi + alpha_j * yj = const = 0 (all others zero, sum=0)
# => alpha_i = -yj/yi * alpha_j  (if yi != 0)
C_demo = 1.0

# Sweep alpha_j over [0, C], compute alpha_i from constraint, evaluate objective
alpha_j_range = np.linspace(0, C_demo, 300)
# Constraint: alpha_i*yi + alpha_j*yj = 0 => alpha_i = -(yj/yi)*alpha_j
alpha_i_range = -(yj / yi) * alpha_j_range
# Only keep valid (both in [0, C])
valid = (alpha_i_range >= 0) & (alpha_i_range <= C_demo)

def dual_obj_pair(ai, aj):
    """Dual objective for just the (i, j) pair (others zero)."""
    return (ai + aj
            - 0.5 * (ai**2 * Kii + aj**2 * Kjj + 2*ai*aj*yi*yj*Kij))

obj_vals = dual_obj_pair(alpha_i_range, alpha_j_range)

# Find the analytical optimum
# From the SMO formula: alpha_j_new = alpha_j + yj*(Ei-Ej)/eta
# Starting from alpha_i=alpha_j=0: Ei = b, Ej = b (both zero)
# => no update from zero. Instead, let's show the objective landscape directly.
best_idx  = np.argmax(obj_vals[valid]) if valid.any() else 0
best_ai   = alpha_i_range[valid][best_idx]
best_aj   = alpha_j_range[valid][best_idx]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: alpha_j on x-axis, objective on y-axis (1D view of constrained opt)
ax = axes[0]
ax.plot(alpha_j_range[valid], obj_vals[valid], color='#3498db', linewidth=2.5)
ax.axvline(best_aj, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Optimum α_j* = {best_aj:.3f}')
ax.scatter([best_aj], [obj_vals[valid][best_idx]], color='#e74c3c', s=120, zorder=5)
ax.set_xlabel(f'α_j (point {j_demo})', fontsize=12)
ax.set_ylabel('Dual Objective W(α_i, α_j)', fontsize=12)
ax.set_title(f'1D Constrained Optimization\n(α_i forced by α_i·y_i + α_j·y_j = 0)',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)

# Right: 2D alpha space showing the constraint line
ax2 = axes[1]
ai_grid = np.linspace(-0.1, C_demo+0.1, 200)
aj_grid = np.linspace(-0.1, C_demo+0.1, 200)
AI, AJ = np.meshgrid(ai_grid, aj_grid)
Z = dual_obj_pair(AI, AJ)

ct = ax2.contourf(AI, AJ, Z, levels=20, cmap='RdYlGn')
plt.colorbar(ct, ax=ax2, label='Objective W')

# Feasible region: 0 <= ai, aj <= C
box = mpatches.FancyArrowPatch((0,0),(C_demo,0), color='gray')
ax2.add_patch(plt.Rectangle((0,0), C_demo, C_demo, fill=False,
                              edgecolor='black', linewidth=2, linestyle='--',
                              label='Feasible box [0,C]²'))

# Constraint line: alpha_i*yi + alpha_j*yj = 0
aj_line = np.linspace(-0.1, C_demo+0.1, 200)
ai_line = -(yj/yi) * aj_line
ax2.plot(aj_line, ai_line, 'w-', linewidth=2.5, label='Constraint line')

# Optimal point
ax2.scatter([best_aj], [best_ai], color='white', s=200, zorder=5,
            edgecolors='black', linewidths=2, label='SMO optimum')

ax2.set_xlabel(f'α_j (point {j_demo})', fontsize=12)
ax2.set_ylabel(f'α_i (point {i_demo})', fontsize=12)
ax2.set_title('2D Alpha Space\nObjective landscape + constraint line',
              fontsize=12, fontweight='bold')
ax2.set_xlim(-0.05, C_demo+0.05)
ax2.set_ylim(-0.05, C_demo+0.05)
ax2.legend(fontsize=9)

plt.suptitle('SMO: The 2-Variable Subproblem Visualized', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Key insight:")
print("The constraint α_i·y_i + α_j·y_j = const reduces the 2D optimization")
print("to a 1D problem along the constraint line — which has a closed-form solution!")
print(f"Analytical optimum: α_j* ≈ {best_aj:.4f}, α_i* ≈ {best_ai:.4f}")

## SMO vs Standard QP: Complexity Summary

| Method | Time per step | # Steps | Total | Memory |
|--------|--------------|---------|-------|--------|
| Standard QP (full) | O(n³) | 1 | **O(n³)** | O(n²) |
| SMO (1 pass) | O(n) per pair | O(n) pairs | **O(n²) per pass** | O(n) |
| SMO (total) | — | — | **O(n) to O(n²) passes** | O(n) |
| LIBSVM (sklearn) | O(1) with caching | — | **O(n²) typical** | O(n·cache) |

**Why SMO is fast:**
1. Each 2-variable update takes O(1) time (closed-form, no inner loop)
2. Only 2 kernel evaluations per update (K_ii, K_jj, K_ij)
3. No kernel matrix storage needed — evaluate on demand
4. Convergence focuses on KKT violators — non-violating points get skipped

**Why SMO is still slow for n > 1M:**
- Even O(n) per pass means 1M kernel evaluations per iteration
- With 10-100 passes needed, that's 10M-100M kernel evaluations
- For large n, use: SGD-SVM, random features + linear SVM, or deep learning

---

## The SMO 2-Variable Constraint: Why Not Update Just One?

The SVM dual requires: $\sum_i \alpha_i y_i = 0$

If we update only $\alpha_k$ (all others fixed):
$$\Delta\alpha_k \cdot y_k + \text{(fixed terms)} = 0 \implies \Delta\alpha_k = 0$$

The only solution is **no change at all**. You literally cannot update a single variable without violating the equality constraint. The minimum size of a feasible update is **2 variables** — which is why SMO is the "minimal" optimization.

---

## KKT Conditions: The Convergence Certificate

When SMO converges (no KKT violations remain), for every training point:
- $\alpha_i = 0$ → point is correctly classified with margin ≥ 1
- $0 < \alpha_i < C$ → point is exactly on the margin (support vector)
- $\alpha_i = C$ → point is inside the margin or misclassified (bounded SV)

This is the **global optimality certificate** for the SVM dual — no further improvement is possible.

In [ ]:
# =============================================================================
# SECTION 14: Final Summary Dashboard
# =============================================================================

smo_final = SimpleSMO(C=1.0, tol=1e-3, max_iter=100, random_state=42)
smo_final.fit(X, y)

fig = plt.figure(figsize=(16, 10))
gs  = fig.add_gridspec(2, 3, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])   # Decision boundary
ax2 = fig.add_subplot(gs[0, 1])   # KKT violations
ax3 = fig.add_subplot(gs[0, 2])   # Dual objective
ax4 = fig.add_subplot(gs[1, 0])   # Alpha convergence
ax5 = fig.add_subplot(gs[1, 1])   # Selection frequency
ax6 = fig.add_subplot(gs[1, 2])   # Changed alphas

# --- 1: Decision boundary ---
z_final = np.array([smo_final._decision_function(pt) for pt in grid_pts]).reshape(xx.shape)
ax1.contourf(xx, yy, z_final, levels=[-np.inf, 0, np.inf],
             colors=['#aed6f1', '#f1948a'], alpha=0.4)
ax1.contour(xx, yy, z_final, levels=[-1,0,1],
            colors=['#2980b9','#2c3e50','#c0392b'], linewidths=[1.5,2.5,1.5])
ax1.scatter(X[y==-1,0], X[y==-1,1], c='#3498db', s=40, edgecolors='black', lw=0.5, zorder=3)
ax1.scatter(X[y==+1,0], X[y==+1,1], c='#e74c3c', s=40, marker='^', edgecolors='black', lw=0.5, zorder=3)
sv_m = smo_final.support_mask_
ax1.scatter(X[sv_m,0], X[sv_m,1], s=180, facecolors='none',
            edgecolors='#f39c12', linewidths=2.5, zorder=4)
ax1.set_title('Decision Boundary\n& Support Vectors (orange ring)', fontsize=10, fontweight='bold')
ax1.set_xlabel('Spam Word Freq', fontsize=9)
ax1.set_ylabel('Sender Reputation', fontsize=9)

# --- 2: KKT violations ---
iters_f = np.arange(1, len(smo_final.history_violations_)+1)
ax2.plot(iters_f, smo_final.history_violations_, 'o-', color='#e74c3c', linewidth=2, markersize=5)
ax2.fill_between(iters_f, smo_final.history_violations_, alpha=0.15, color='#e74c3c')
ax2.set_title('KKT Violations → 0', fontsize=10, fontweight='bold')
ax2.set_xlabel('Iteration', fontsize=9)
ax2.set_ylabel('Violations', fontsize=9)

# --- 3: Dual objective ---
ax3.plot(iters_f, smo_final.history_objective_, 's-', color='#2ecc71', linewidth=2, markersize=5)
ax3.set_title('Dual Objective ↑ to Optimum', fontsize=10, fontweight='bold')
ax3.set_xlabel('Iteration', fontsize=9)
ax3.set_ylabel('W(α)', fontsize=9)

# --- 4: Alpha convergence for SVs ---
sv_idx_f = np.where(smo_final.support_mask_)[0]
cols_f   = plt.cm.Set1(np.linspace(0, 1, len(sv_idx_f)))
for sv_i_f, c_f in zip(sv_idx_f, cols_f):
    ax4.plot(iters_f, smo_final.alpha_history_[:, sv_i_f], '-', color=c_f, linewidth=2)
ax4.set_title('α Values Converging\n(only final SVs shown)', fontsize=10, fontweight='bold')
ax4.set_xlabel('Iteration', fontsize=9)
ax4.set_ylabel('α value', fontsize=9)
ax4.axhline(smo_final.C, color='gray', linestyle='--', linewidth=1, alpha=0.6)

# --- 5: Selection frequency ---
pairs_f = smo_final.pairs_selected_
psc     = np.zeros(len(X))
for (pi, pj) in pairs_f:
    psc[pi] += 1
    psc[pj] += 1

colors_pts5 = ['#f39c12' if smo_final.support_mask_[k] else '#bdc3c7' for k in range(len(X))]
ax5.bar(range(len(X)), psc, color=colors_pts5, edgecolor='none', width=0.8)
ax5.set_title('Selection Frequency per Point\n(orange = final SV)', fontsize=10, fontweight='bold')
ax5.set_xlabel('Training Point Index', fontsize=9)
ax5.set_ylabel('Times selected', fontsize=9)

# --- 6: Changed alphas per iteration ---
ax6.bar(iters_f, smo_final.history_changed_, color='#9b59b6', alpha=0.8, edgecolor='white')
ax6.set_title('Changed Alphas per Iteration\n(→0 at convergence)', fontsize=10, fontweight='bold')
ax6.set_xlabel('Iteration', fontsize=9)
ax6.set_ylabel('# Changed', fontsize=9)

fig.suptitle('SMO Algorithm — Complete Summary Dashboard', fontsize=15, fontweight='bold')
plt.show()

print("SMO Summary:")
print(f"  Training accuracy:  {accuracy_score(y, smo_final.predict(X)):.4f}")
print(f"  Support vectors:    {smo_final.n_support_}")
print(f"  Iterations to converge: {len(smo_final.history_changed_)}")
print(f"  Final KKT violations:   {smo_final.history_violations_[-1]}")
print(f"  sum(alpha_i * y_i):     {np.dot(smo_final.alpha_, y):.8f}  (≈ 0 ✓)")

## Self-Check Questions

Test your understanding before moving on.

---

**Question 1:** SMO updates exactly 2 alphas at a time. Why can't it update just 1? *(Hint: what constraint would be violated?)*

<details>
<summary>Click to reveal answer</summary>

**Because the SVM dual has an equality constraint $\sum_i \alpha_i y_i = 0$ that must be maintained at all times.**

Suppose all other $\alpha_k$ (k ≠ i) are fixed. The constraint becomes:
$$\alpha_i y_i + \underbrace{\sum_{k \neq i} \alpha_k y_k}_{\text{fixed constant}} = 0$$

This means $\alpha_i$ is **completely determined** by all the other alphas — it has zero degrees of freedom! Any change to $\alpha_i$ alone would violate the constraint. The only valid update is $\Delta\alpha_i = 0$ (no change).

By updating **two** variables simultaneously $(\alpha_i, \alpha_j)$, we have one degree of freedom: we can increase one and decrease the other while keeping $\alpha_i y_i + \alpha_j y_j = \text{constant}$ (and thus keeping the full sum zero). This is the **minimum number** of variables that can be jointly updated while maintaining the equality constraint — hence "Minimal" in Sequential Minimal Optimization.

**Budget analogy:** You cannot add money to one department alone. You must simultaneously remove the same amount from another department to keep the total budget at \$10M.

</details>

---

**Question 2:** SMO converges when there are no more KKT violations. After convergence, what do you know about the relationship between each $\alpha_i$ and the training point $x_i$?

<details>
<summary>Click to reveal answer</summary>

**After convergence, the KKT conditions fully characterize the role of each training point:**

| $\alpha_i$ value | KKT condition | Meaning for point $x_i$ |
|-----------------|--------------|-------------------------|
| $\alpha_i = 0$ | $y_i f(x_i) \geq 1$ | Correctly classified, outside (or on) the margin. Doesn't contribute to the decision boundary. |
| $0 < \alpha_i < C$ | $y_i f(x_i) = 1$ | Exactly on the margin — a **free support vector**. These points define the margin width and the bias b. |
| $\alpha_i = C$ | $y_i f(x_i) \leq 1$ | Inside the margin or misclassified — a **bounded support vector**. These are the "hardest" points (violations of the hard margin). |

This is the **complementary slackness** from KKT theory:
- $\alpha_i = 0 \Rightarrow$ slack constraint is strict (point is well inside margin)
- $\alpha_i > 0 \Rightarrow$ constraint is active (point is on or inside the margin)

Practically: after convergence, you can identify every SV by checking $\alpha_i > 0$, and you can read the margin from the free SVs ($0 < \alpha_i < C$) where $y_i f(x_i) = 1$ exactly.

</details>

---

**Question 3:** Standard QP for SVM is O(n³). SMO is O(n) to O(n²) per pass. Why is SMO still slow on very large datasets (n > 1 million)?

<details>
<summary>Click to reveal answer</summary>

**Several reasons why SMO struggles at n > 1M:**

1. **Kernel evaluations still dominate:** Even if each SMO step is O(1), finding the best pair (working set selection) requires computing errors $E_i = f(x_i) - b$ for all n points. This needs $n \times n_{SV}$ kernel evaluations per outer pass — if there are many SVs, this is O(n²) per pass.

2. **Number of passes can grow:** For noisy or complex datasets, SMO may need O(n) passes to converge, giving O(n³) worst-case total — the same as full QP, just with a much smaller constant.

3. **Cache misses:** Even with an LRU kernel cache (as in LIBSVM), with n=1M the cache hit rate drops dramatically. Each kernel evaluation K(x_i, x_j) requires computing the dot product of two n_features-dimensional vectors — with a million points and hundreds of features, this doesn't fit in CPU cache.

4. **Memory for alpha and error arrays:** Storing n alpha values and n error values requires O(n) memory — 1M doubles = 8 MB each, still manageable. But the kernel matrix itself (if stored) is O(n²) = **8 TB for n=1M** — impossible.

**Solutions for large-scale SVMs:**
- **Pegasos / SGD-SVM:** Stochastic gradient descent on the primal — O(1/ε) iterations, truly linear in n
- **Kernel approximations:** Nyström approximation or Random Fourier Features reduce kernel computation to a finite-dimensional feature map, allowing standard linear SGD
- **LinearSVC (sklearn):** Uses LIBLINEAR (coordinate descent on the dual) — O(n) per pass, excellent for n > 100K
- **Deep learning:** For very large datasets, neural networks often outperform SVMs and scale better with modern hardware

</details>

In [ ]:
# =============================================================================
# BONUS: SMO on Non-Linearly Separable Data (with C slack)
# =============================================================================
# Show that SMO with soft margin (bounded alphas) handles overlapping classes.

np.random.seed(42)
n_bonus = 60

# Overlapping classes
X_b = np.vstack([
    np.random.normal([-1.0, 1.0], 0.9, (n_bonus//2, 2)),
    np.random.normal([ 1.0,-1.0], 0.9, (n_bonus//2, 2))
])
y_b = np.array([-1]*(n_bonus//2) + [1]*(n_bonus//2), dtype=float)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
C_bonus_vals = [0.1, 1.0, 10.0]
colors_b     = ['#3498db', '#e67e22', '#e74c3c']

x_min_b = X_b[:,0].min()-0.5
x_max_b = X_b[:,0].max()+0.5
y_min_b = X_b[:,1].min()-0.5
y_max_b = X_b[:,1].max()+0.5
xx_b, yy_b = np.meshgrid(np.linspace(x_min_b, x_max_b, 150),
                           np.linspace(y_min_b, y_max_b, 150))
grid_b = np.column_stack([xx_b.ravel(), yy_b.ravel()])

for ax, C_b, col_b in zip(axes, C_bonus_vals, colors_b):
    smo_b = SimpleSMO(C=C_b, tol=1e-3, max_iter=50, random_state=42)
    smo_b.fit(X_b, y_b)

    z_b = np.array([smo_b._decision_function(pt) for pt in grid_b]).reshape(xx_b.shape)
    acc_b = accuracy_score(y_b, smo_b.predict(X_b))

    ax.contourf(xx_b, yy_b, z_b, levels=[-np.inf,0,np.inf],
                colors=['#aed6f1','#f1948a'], alpha=0.35)
    ax.contour(xx_b, yy_b, z_b, levels=[0], colors=[col_b], linewidths=2.5)

    sv_m_b = smo_b.support_mask_
    ax.scatter(X_b[y_b==-1,0], X_b[y_b==-1,1], c='#3498db', s=50,
               edgecolors='black', lw=0.5, zorder=3)
    ax.scatter(X_b[y_b==+1,0], X_b[y_b==+1,1], c='#e74c3c', s=50, marker='^',
               edgecolors='black', lw=0.5, zorder=3)
    ax.scatter(X_b[sv_m_b,0], X_b[sv_m_b,1], s=180, facecolors='none',
               edgecolors='#f39c12', linewidths=2.5, zorder=4,
               label=f'SVs ({sv_m_b.sum()})')

    ax.set_title(
        f'C = {C_b}  |  SVs = {sv_m_b.sum()}\n'
        f'Accuracy = {acc_b:.3f}',
        fontsize=12, fontweight='bold', color=col_b
    )
    ax.set_xlabel('Spam Word Frequency', fontsize=10)
    ax.set_ylabel('Sender Reputation', fontsize=10)
    ax.legend(fontsize=9)

plt.suptitle('SMO on Non-Linearly Separable Data — Effect of C (Soft Margin)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("  Small C (0.1): Wide margin, many points inside/misclassified (many bounded SVs)")
print("  Large C (10.0): Narrow margin, tries to classify everything correctly")
print("  SMO handles both cases via the clipping step: alpha_j ∈ [L, H] where H ≤ C")

In [ ]:
# =============================================================================
# SECTION 15: Numerical Verification of the KKT Conditions at Convergence
# =============================================================================
# After SMO converges, every alpha_i should satisfy KKT conditions.
# Let's verify this numerically and display a clean summary table.

print("KKT Condition Verification at Convergence")
print("="*60)
print(f"Tolerance: tol = {smo_final.tol}")
print()

K_full = X @ X.T  # linear kernel matrix
f_vals = K_full @ (smo_final.alpha_ * y) + smo_final.b_  # f(x_i) for all i

rows = []
for i in range(len(X)):
    a_i  = smo_final.alpha_[i]
    yf_i = y[i] * f_vals[i]

    # Determine KKT category
    if a_i < smo_final.tol:
        category = 'alpha=0  (interior)'
        kkt_ok   = yf_i >= 1 - smo_final.tol
    elif a_i > smo_final.C - smo_final.tol:
        category = 'alpha=C  (bounded SV)'
        kkt_ok   = yf_i <= 1 + smo_final.tol
    else:
        category = '0<alpha<C (free SV)'
        kkt_ok   = abs(yf_i - 1) <= smo_final.tol

    rows.append({'i': i, 'alpha_i': round(a_i, 5), 'y_i*f(x_i)': round(yf_i, 5),
                 'category': category, 'KKT OK': kkt_ok})

df_kkt = pd.DataFrame(rows)

# Summary counts
print(df_kkt.groupby('category')[['KKT OK']].agg(['count', 'sum']).to_string())
print()
print(f"Total points:         {len(X)}")
print(f"KKT satisfied:        {df_kkt['KKT OK'].sum()} / {len(X)}")
print(f"KKT violated:         {(~df_kkt['KKT OK']).sum()} / {len(X)}")
print()

# Show the support vectors with their KKT status
sv_df = df_kkt[df_kkt['alpha_i'] > smo_final.tol].copy()
print("Support Vectors (alpha > 0):")
print(sv_df[['i', 'alpha_i', 'y_i*f(x_i)', 'category', 'KKT OK']].to_string(index=False))
print()

# Visualize KKT margin values
fig, ax = plt.subplots(figsize=(12, 5))
interior = df_kkt['category'].str.contains('interior')
bounded  = df_kkt['category'].str.contains('bounded')
free_sv  = df_kkt['category'].str.contains('free')

ax.scatter(df_kkt.index[interior], df_kkt['y_i*f(x_i)'][interior],
           color='#3498db', s=40, alpha=0.6, label='alpha=0 (interior)', zorder=3)
ax.scatter(df_kkt.index[bounded], df_kkt['y_i*f(x_i)'][bounded],
           color='#e74c3c', s=80, marker='s', edgecolors='black', lw=0.7,
           label='alpha=C (bounded SV)', zorder=4)
ax.scatter(df_kkt.index[free_sv], df_kkt['y_i*f(x_i)'][free_sv],
           color='#f39c12', s=100, marker='^', edgecolors='black', lw=0.7,
           label='0<alpha<C (free SV)', zorder=5)

ax.axhline(1.0, color='green',  linestyle='--', linewidth=2.0, alpha=0.8, label='y*f(x) = 1 (margin)')
ax.axhline(1.0 + smo_final.tol, color='green', linestyle=':', linewidth=1.2, alpha=0.4)
ax.axhline(1.0 - smo_final.tol, color='green', linestyle=':', linewidth=1.2, alpha=0.4)
ax.set_xlabel('Training Point Index', fontsize=12)
ax.set_ylabel('y_i * f(x_i)  (functional margin)', fontsize=12)
ax.set_title('KKT Verification: Functional Margins at Convergence\n'
             'Interior: y*f(x) >= 1   |   Free SV: y*f(x) = 1   |   Bounded SV: y*f(x) <= 1',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(ax.get_ylim()[0] - 0.1, ax.get_ylim()[1] + 0.1)
plt.tight_layout()
plt.show()

print("KKT interpretation:")
print("  Interior points (alpha=0):   y*f(x) >= 1  -> correctly classified outside margin")
print("  Free SVs (0<alpha<C):        y*f(x) = 1   -> exactly on the margin")
print("  Bounded SVs (alpha=C):       y*f(x) <= 1  -> inside margin or misclassified")

## Key Takeaways

1. **SMO decomposes the SVM QP into 2-variable subproblems** — the minimum size needed to maintain the equality constraint $\sum \alpha_i y_i = 0$. Each subproblem has a closed-form solution.

2. **Working set selection is critical:** SMO picks the pair $(\alpha_i, \alpha_j)$ that (a) violates KKT and (b) maximizes $|E_i - E_j|$. This heuristic ensures large updates and fast convergence.

3. **KKT violations measure progress:** At convergence, zero KKT violations = global optimality certificate. KKT conditions fully characterize every $\alpha_i$: zero (interior), free (on margin), or bounded (inside/misclassified).

4. **SMO is O(n) to O(n²) per pass** vs O(n³) for full QP. For n=1M, SMO is still slow — use SGD-SVM or kernel approximations instead.

5. **SimpleSMO vs sklearn SVC:** Our Python implementation gives the same decision boundary. sklearn's LIBSVM is a C++ implementation of the same algorithm with optimizations (kernel caching, better working set selection, shrinking heuristic).

6. **The constraint $\alpha_i \in [0, C]$** encodes the soft margin: $\alpha_i = C$ means the point is a bounded SV (inside or beyond the margin). The clipping step in SMO enforces this box constraint analytically.

---

**You have now completed Week 10: Support Vector Machines & Kernel Methods.**

What you've learned:
- Maximum margin classification (Notebook 1)
- Hard & soft margin SVMs (Notebooks 2-3)
- The kernel trick (Notebooks 4-5)
- Multi-class SVM (Notebook 6)
- Probabilistic SVM (Notebook 7)
- SVM for structured outputs (Notebook 8)
- Support Vector Regression (Notebook 9)
- SMO Algorithm (Notebook 10 — this one)

**Next week:** Ensemble Methods — Bagging, Boosting, and Random Forests